# Dataset Raw de LaLiga - EDA

Análisis Exploratorio de Datos (EDA) de las temporadas de LaLiga (`2014-15` a `2023-24`) a partir de datasets de partidos en formato raw procedentes de `football-data.co.uk`.

## Objetivos
- Comprender la estructura del dataset y la estabilidad del esquema entre temporadas.
- Evaluar la calidad e integridad de los datos (valores nulos, duplicados y rangos inválidos).
- Identificar posibles riesgos antes de avanzar a la fase de limpieza de datos.

## Estructura del Notebook

El análisis se organiza en las siguientes secciones:

**0. Entorno y configuración**  
Carga de librerías y definición de rutas del proyecto.

**1. Preparación y organización de datos**  
Identificación de ficheros disponibles y construcción del diccionario de temporadas.

**2. Consistencia del esquema de datos**  
Comparación de dimensiones, definición del core dataset y verificación de estabilidad en tipos de dato.

**3. Calidad de datos**  
Evaluación de la completitud del core dataset por variable y temporada.

**4. Validaciones de integridad**  
Comprobaciones de coherencia en el core dataset (resultados, nombres de equipos, duplicados y valores no válidos).

**5. Variables de cuotas**  
Identificación de columnas de apuestas, análisis de cobertura y validación de rangos.

**6. Conclusiones del análisis exploratorio**  
Síntesis estructural y consideraciones para la fase de limpieza.

## 0) Entorno y configuración

En esta sección se configuran las dependencias, librerías y parámetros globales necesarios para garantizar la reproducibilidad del análisis.

In [31]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Configuración de rutas del proyecto
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Importación de utilidades propias
from src.analysis import check_name_consistency, group_columns


## 1) Preparación y organización de datos

En esta fase se identifican y organizan los datos por temporada con el fin de establecer una estructura coherente previa al análisis estructural.


### 1.1 Carga de datos y verificación inicial

Se identifican los archivos CSV disponibles y se comprueba la disponibilidad de temporadas suficientes para el análisis.

In [2]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "laliga"
csv_files = sorted(RAW_DIR.glob('laliga_*_raw.csv'))

if len(csv_files) < 2:
    raise ValueError(f'Se esperaban al menos 2 ficheros en {RAW_DIR}, encontrados {len(csv_files)}')

print(f"Directorio analizado: {RAW_DIR}")
print(f"Ficheros detectados: {len(csv_files)}")

Directorio analizado: /Users/jorgepais/Desktop/kraken/football-analytics/data/raw/laliga
Ficheros detectados: 10


### 1.2 Inspección de ficheros disponibles

Se listan los ficheros detectados para verificar las temporadas disponibles antes de su carga.

In [3]:
names = [p.name for p in csv_files]

HEAD_N = 5
print("Muestra de ficheros:")

if len(names) <= 2 * HEAD_N:
    for n in names:
        print(f"  - {n}")
else:
    for n in names[:HEAD_N]:
        print(f"  - {n}")
    print(f"  ... ({len(names) - 2 * HEAD_N} ficheros omitidos) ...")
    for n in names[-HEAD_N:]:
        print(f"  - {n}")

Muestra de ficheros:
  - laliga_2014_15_raw.csv
  - laliga_2015_16_raw.csv
  - laliga_2016_17_raw.csv
  - laliga_2017_18_raw.csv
  - laliga_2018_19_raw.csv
  - laliga_2019_20_raw.csv
  - laliga_2020_21_raw.csv
  - laliga_2021_22_raw.csv
  - laliga_2022_23_raw.csv
  - laliga_2023_24_raw.csv


### 1.3 Construcción del diccionario de temporadas

Se construye un diccionario que asocia cada temporada con su correspondiente DataFrame.

In [4]:
dfs_all = {}

for f in csv_files:
    season = f.stem.replace('_raw', '').replace('laliga_', '')  
    dfs_all[season] = pd.read_csv(f)

seasons_sorted = sorted(
    dfs_all.keys(),
    key=lambda s: int(s.split('_')[0])
)

n = len(seasons_sorted)
print(f"Temporadas cargadas: {n}  ({seasons_sorted[0]} → {seasons_sorted[-1]})\n")
print("  |  ".join(seasons_sorted))

Temporadas cargadas: 10  (2014_15 → 2023_24)

2014_15  |  2015_16  |  2016_17  |  2017_18  |  2018_19  |  2019_20  |  2020_21  |  2021_22  |  2022_23  |  2023_24


## 2) Consistencia del esquema de datos 

En esta fase se analiza la coherencia en la estructura del dataset entre temporadas, evaluando la estabilidad de sus dimensiones, variables y tipos de dato.

### 2.1 Dimensiones del dataset por temporada

Se comparan filas y columnas de cada temporada para identificar posibles cambios estructurales.

In [5]:
summary = pd.DataFrame(
    [{
        "Season": s,
        "Rows": dfs_all[s].shape[0],
        "Columns": dfs_all[s].shape[1],
    } for s in seasons_sorted]
)

display(summary.style.hide(axis="index"))

print(f"Filas consistentes: {summary['Rows'].nunique() == 1}")
print(f"Columnas consistentes: {summary['Columns'].nunique() == 1}")

Season,Rows,Columns
2014_15,380,67
2015_16,380,64
2016_17,380,64
2017_18,380,64
2018_19,380,61
2019_20,380,105
2020_21,380,105
2021_22,380,105
2022_23,380,105
2023_24,380,105


Filas consistentes: True
Columnas consistentes: False


### 2.2 Intersección de columnas comunes

Se obtiene la intersección de columnas comunes a todas las temporadas.

In [6]:
column_sets = [set(dfs_all[s].columns) for s in seasons_sorted]
common_columns = set.intersection(*column_sets)
print(f"Número de columnas comunes a TODAS las temporadas: {len(common_columns)}")

Número de columnas comunes a TODAS las temporadas: 43


### 2.3 Conjunto de variables comunes (core dataset)

Listado de variables comunes a todo el histórico de temporadas.

#### Resumen global

In [7]:
core_df = pd.DataFrame(sorted(common_columns), columns=["Variable"])
df_class = group_columns(core_df["Variable"])

summary_groups = (
    df_class.groupby("Grupo", as_index=False)
    .agg(**{"Número de variables": ("Variable", "count")})
    .sort_values("Número de variables", ascending=False)
)

display(summary_groups.style.hide(axis="index"))

Grupo,Número de variables
Cuotas de apuestas,21
Estadísticas del partido,12
Resultados y goles,6
Identificación del partido,4


#### Desagregado por grupos

In [8]:
detail_groups = (
    df_class.sort_values(["Grupo", "Variable"])
    .groupby("Grupo", as_index=False)
    .agg(Variables=("Variable", lambda x: "\n".join(x)))
)

display(
    detail_groups.style
    .set_properties(**{"white-space": "pre-wrap"})
    .hide(axis="index")
)

Grupo,Variables
Cuotas de apuestas,B365A B365D B365H BWA BWD BWH IWA IWD IWH PSA PSCA PSCD PSCH PSD PSH VCA VCD VCH WHA WHD WHH
Estadísticas del partido,AC AF AR AS AST AY HC HF HR HS HST HY
Identificación del partido,AwayTeam Date Div HomeTeam
Resultados y goles,FTAG FTHG FTR HTAG HTHG HTR


### 2.4 Tipos de datos del core dataset

Clasificación de las variables comunes según su naturaleza: numéricas, categóricas y temporales.

In [9]:
sample_df = dfs_all[seasons_sorted[0]][sorted(common_columns)]

numeric_cols = sorted(sample_df.select_dtypes(include=['number']).columns.tolist())
categorical_cols = sorted(sample_df.select_dtypes(include=['object', 'string']).columns.tolist())
datetime_cols = sorted(sample_df.select_dtypes(include=['datetime']).columns.tolist())

type_summary = pd.DataFrame({
    "Tipo": ["Numéricas", "Categóricas", "Temporales"],
    "Cantidad": [len(numeric_cols), len(categorical_cols), len(datetime_cols)],
    "Variables": [
        ", ".join(numeric_cols) if numeric_cols else "-",
        ", ".join(categorical_cols) if categorical_cols else "-",
        ", ".join(datetime_cols) if datetime_cols else "-"
    ]
})

display(type_summary.style.hide(axis="index"))
print(f"\nMuestra de datos de la temporada {seasons_sorted[0]} (primeras 5 filas):")
dfs_all[seasons_sorted[0]].head(5)

Tipo,Cantidad,Variables
Numéricas,37,"AC, AF, AR, AS, AST, AY, B365A, B365D, B365H, BWA, BWD, BWH, FTAG, FTHG, HC, HF, HR, HS, HST, HTAG, HTHG, HY, IWA, IWD, IWH, PSA, PSCA, PSCD, PSCH, PSD, PSH, VCA, VCD, VCH, WHA, WHD, WHH"
Categóricas,6,"AwayTeam, Date, Div, FTR, HTR, HomeTeam"
Temporales,0,-



Muestra de datos de la temporada 2014_15 (primeras 5 filas):


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
0,SP1,23/08/14,Almeria,Espanol,1,1,D,0,0,D,...,1.64,26,0.0,1.98,1.91,2.00,1.94,3.19,3.26,2.50
1,SP1,23/08/14,Granada,La Coruna,2,1,H,0,1,A,...,1.54,24,-0.5,2.11,2.04,1.88,1.84,2.20,3.34,3.75
2,SP1,23/08/14,Malaga,Ath Bilbao,1,0,H,1,0,H,...,1.77,26,0.0,2.09,2.03,1.88,1.83,3.14,3.23,2.56
3,SP1,23/08/14,Sevilla,Valencia,1,1,D,1,0,H,...,1.92,23,-0.5,2.10,2.03,1.90,1.84,2.29,3.46,3.40
4,SP1,24/08/14,Barcelona,Elche,3,0,H,1,0,H,...,3.48,27,-2.5,1.89,1.77,2.18,2.10,1.11,12.80,24.00


### 2.5 Cambios en tipos de datos

Detección de columnas cuyo `dtype` (tipo de dato) cambia según la temporada.

In [10]:
dtype_rows = []

for s in seasons_sorted:
    df = dfs_all[s]
    dtypes = df.dtypes.astype(str)

    for col, dtype in dtypes.items():
        dtype_rows.append({
            "Column": col,
            "Season": s,
            "dtype": dtype
        })

dtype_df = pd.DataFrame(dtype_rows)
dtype_pivot = dtype_df.pivot(index="Column", columns="Season", values="dtype")
type_counts = dtype_pivot.nunique(axis=1)
dtype_inconsistent = dtype_pivot[type_counts > 1]

print(f"Número de columnas con cambios en dtype: {dtype_inconsistent.shape[0]}")
display(dtype_inconsistent)

Número de columnas con cambios en dtype: 0


Season,2014_15,2015_16,2016_17,2017_18,2018_19,2019_20,2020_21,2021_22,2022_23,2023_24
Column,,,,,,,,,,


## 3) Calidad de datos: valores nulos

Se analiza la presencia de valores nulos por variable y temporada en el core dataset.

### 3.1 Construcción del resumen de valores nulos

Se consolida la información de valores nulos en una tabla estructurada por temporada y variable.

In [23]:
null_rows = []

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]
    null_counts = df.isna().sum()
    null_pct = (df.isna().mean() * 100).round(2)
    
    season_nulls = pd.DataFrame({
        "Season": s,
        "Column": df.columns,
        "Nulls": null_counts.values,
        "Null_%": null_pct.values
    })
    null_rows.append(season_nulls)

null_core_df = pd.concat(null_rows, ignore_index=True)
print(f"Nulos detectados en el core dataset: {null_core_df['Nulls'].sum():,}")

Nulos detectados en el core dataset: 633


### 3.2 Columnas con nulos relevantes

Se agregan los resultados para obtener una visión global del nivel de ausencia de datos en cada temporada.

In [24]:
season_summary = (
    null_core_df.groupby("Season", as_index=False)
    .agg(
        **{
            "% nulos medio": ("Null_%", "mean"),
            "Columnas con nulos": ("Nulls", lambda x: (x > 0).sum()),
            "Máx. % nulos": ("Null_%", "max"),
        }
    )
)

display(season_summary.style.hide(axis="index"))

Season,% nulos medio,Columnas con nulos,Máx. % nulos
2014_15,0.055116,6,0.530000
2015_16,0.018140,3,0.260000
2016_17,0.055116,3,0.790000
2017_18,0.018140,3,0.260000
2018_19,0.000000,0,0.000000
2019_20,0.036977,3,0.530000
2020_21,0.000000,0,0.000000
2021_22,0.018140,3,0.260000
2022_23,0.000000,0,0.000000
2023_24,3.671860,6,50.000000


### 3.3 Filtro de nulos relevantes

Se identifican las variables cuyo porcentaje de nulos supera el umbral definido.

In [25]:
THRESHOLD_NULL_PCT = 5.0

null_relevant = (
    null_core_df[null_core_df["Null_%"] > THRESHOLD_NULL_PCT]
    .sort_values(["Season", "Null_%"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"Umbral aplicado: > {THRESHOLD_NULL_PCT:.1f}% nulos")
print(f"Filas que superan el umbral: {len(null_relevant)}")

if len(null_relevant) > 0:
    display(null_relevant.style.hide(axis="index"))
else:
    print("Ninguna columna del core supera el umbral")

Umbral aplicado: > 5.0% nulos
Filas que superan el umbral: 3


Season,Column,Nulls,Null_%
2023_24,IWA,190,50.000000
2023_24,IWD,190,50.000000
2023_24,IWH,190,50.000000


### 3.4 Análisis global por variable

Se detectan las variables que presentan mayores niveles de ausencia de datos en el core dataset.

In [26]:
worst_columns = (
    null_core_df.groupby("Column", as_index=False)
    .agg(**{"Máx. % nulos (cualquier temporada)": ("Null_%", "max")})
    .sort_values("Máx. % nulos (cualquier temporada)", ascending=False)
)

worst_columns_filtered = worst_columns[worst_columns["Máx. % nulos (cualquier temporada)"] > 0]

if len(worst_columns_filtered) > 0:
    display(worst_columns_filtered.style.hide(axis="index"))
else:
    print("Ninguna columna del core presenta nulos en ninguna temporada")

Column,Máx. % nulos (cualquier temporada)
IWA,50.000000
IWD,50.000000
IWH,50.000000
BWD,2.630000
BWA,2.630000
BWH,2.630000
PSA,0.530000
PSH,0.530000
PSD,0.530000
PSCH,0.260000


## 4) Validaciones de integridad

Se realizan comprobaciones básicas de coherencia lógica y estructural sobre el core dataset.


### 4.1 Validación de categorías en resultados

Se comprueba que las variables `FTR` (resultado final) y `HTR` (resultado al descanso) contengan únicamente las categorías esperadas: `H`, `D` y `A`.

In [29]:
expected = {"H", "D", "A"}
issues_found = False

for s in seasons_sorted:
    ftr_vals = set(dfs_all[s]["FTR"].dropna().unique())
    htr_vals = set(dfs_all[s]["HTR"].dropna().unique())
    
    if ftr_vals != expected or htr_vals != expected:
        print(f"[!] {s}: FTR={ftr_vals}, HTR={htr_vals}")
        issues_found = True

if not issues_found:
    print("Validación FTR/HTR: todas las temporadas correctas")

Validación FTR/HTR: todas las temporadas correctas


### 4.2 Consistencia en nombres de equipos

Se valida la consistencia de los nombres de equipos a lo largo de las temporadas del core dataset.

In [30]:
result = check_name_consistency(dfs_all, seasons_sorted, common_columns)

print(f"Equipos únicos: {result['total_teams']}  |  Colisiones: {len(result['collisions'])}\n")

if len(result['collisions']) > 0:
    print("[!] Colisiones detectadas:")
    display(result['collisions'][["Team_norm", "n_variants", "Variants"]].style.hide(axis="index"))
else:
    print("Nombres de equipos consistentes entre temporadas")

teams_by_season = {}
for s in seasons_sorted:
    df_core = dfs_all[s][sorted(common_columns)]
    teams_by_season[s] = set(df_core["HomeTeam"].unique()) | set(df_core["AwayTeam"].unique())

all_teams = sorted(set(team for teams in teams_by_season.values() for team in teams))
teams_df = pd.DataFrame({"Equipo": all_teams})
display(teams_df.style.hide(axis="index").set_table_attributes('style="max-height:300px; overflow-y:auto; display:block;"'))

Equipos únicos: 31  |  Colisiones: 0

Nombres de equipos consistentes entre temporadas


Equipo
Alaves
Almeria
Ath Bilbao
Ath Madrid
Barcelona
Betis
Cadiz
Celta
Cordoba
Eibar


### 4.3 Detección de duplicados

Se detectan posibles partidos duplicados utilizando `Date`, `HomeTeam` y `AwayTeam` como identificador del mismo.

In [37]:
duplicates_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]
    dups = df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"], keep=False)
    
    if dups.sum() > 0:
        print(f"[!] {s}: {dups.sum()} filas duplicadas detectadas")
        display(df[dups][["Date", "HomeTeam", "AwayTeam", "FTR"]])
        duplicates_found = True

if not duplicates_found:
    print("Validación de duplicados: ninguna temporada afectada")

Validación de duplicados: ninguna temporada afectada


### 4.4 Coherencia resultado vs goles

Validación de consistencia entre `FTR` y marcadores finales (`FTHG`, `FTAG`).

In [38]:
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]

    inconsistent = df[
        ((df["FTR"] == "H") & (df["FTHG"] <= df["FTAG"])) |
        ((df["FTR"] == "A") & (df["FTAG"] <= df["FTHG"])) |
        ((df["FTR"] == "D") & (df["FTHG"] != df["FTAG"]))
    ]

    if len(inconsistent) > 0:
        print(f"[!] {s}: {len(inconsistent)} inconsistencias FTR vs goles")
        display(inconsistent[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]])
        issues_found = True

if not issues_found:
    print("Validación de consistencia FTR: todas las temporadas correctas")

Validación de consistencia FTR: todas las temporadas correctas


### 4.5 Control de valores negativos

Control de calidad para columnas numéricas del core dataset donde no se esperan valores negativos.

In [24]:
numeric_sets = [
    set(dfs_all[s][sorted(common_columns)].select_dtypes(include="number").columns)
    for s in seasons_sorted
]

core_numeric = sorted(set.intersection(*numeric_sets))

core_numeric_df = pd.DataFrame(core_numeric, columns=["Variable numérica estable"])

print(f"Columnas numéricas estables detectadas: {len(core_numeric)}")
display(
    core_numeric_df
    .style
    .hide(axis="index")
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .set_table_attributes('style="max-height:300px; overflow-y:auto; display:block;"')
)

Columnas numéricas estables detectadas: 37


Variable numérica estable
AC
AF
AR
AS
AST
AY
B365A
B365D
B365H
BWA


In [25]:
negative_summary = []
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s]

    for col in core_numeric:
        neg_mask = df[col] < 0
        neg_count = neg_mask.sum()
        
        negative_summary.append({
            "Season": s,
            "Column": col,
            "Negative_values": neg_count
        })
        
        if neg_count > 0:
            print(f"[!] {s} - {col}: {neg_count} valores negativos")
            display(df[neg_mask][["Date", "HomeTeam", "AwayTeam", col]])
            issues_found = True

if not issues_found:
    print("Validación de valores negativos: todas las columnas correctas")

negative_df = pd.DataFrame(negative_summary)

Validación de valores negativos: todas las columnas correctas


### 4.6 Resumen de negativos detectados

Se presenta una vista filtrada de las métricas que contienen valores negativos para facilitar su revisión.

In [47]:
problematic = negative_df[negative_df["Negative_values"] > 0]

if len(problematic) > 0:
    display(problematic.style.hide(axis="index"))
else:
    print("Resumen: ninguna columna contiene valores negativos")

Resumen: ninguna columna contiene valores negativos


## 5) Variables de cuotas

Análisis de columnas asociadas a casas de apuestas para verificar su consistencia, integridad y cobertura en el dataset.

### 5.1 Identificación de columnas de casas de apuestas 

Se identifican las columnas de casas de apuestas presentes en todas las temporadas del dataset.

In [26]:
# Prefijos de casas de apuestas según documentación oficial de football-data
bookmaker_prefixes = [
    "1XB", "B365", "BF", "BFD", "BMGM", "BV", "BS", "BW", 
    "CL", "GB", "IW", "LB", "PS", "P", "SO", "SB", "SJ", 
    "SY", "VC", "WH"
]

odds_columns = [col for col in common_columns 
                if any(col.startswith(book) for book in bookmaker_prefixes)]

if odds_columns:
    bookmakers = {}
    for col in odds_columns:
        for book in bookmaker_prefixes:
            if col.startswith(book):
                bookmakers.setdefault(book, []).append(col)
                break
    
    print(f"Casas de apuestas estables: {len(bookmakers)}  |  Total columnas: {len(odds_columns)}\n")
    
    for book, cols in sorted(bookmakers.items()):
        print(f"  {book:6s} → {', '.join(sorted(cols))}")
else:
    print("No se detectaron columnas de odds estables en todas las temporadas")

Casas de apuestas estables: 6  |  Total columnas: 21

  B365   → B365A, B365D, B365H
  BW     → BWA, BWD, BWH
  IW     → IWA, IWD, IWH
  PS     → PSA, PSCA, PSCD, PSCH, PSD, PSH
  VC     → VCA, VCD, VCH
  WH     → WHA, WHD, WHH


### 5.2 Consistencia de mercados por casa de apuestas

Verificación de que cada casa de apuestas tiene las tres columnas esperadas (H/D/A).

In [27]:
issues_found = False
extra_variants = []

for book in bookmakers:
    cols = set(bookmakers[book])
    
    basic = {f"{book}H", f"{book}D", f"{book}A"}
    if not basic.issubset(cols):
        print(f"[!] {book}: mercado básico H/D/A incompleto")
        issues_found = True
    elif len(cols) > 3:
        extra_variants.append(f"{book} ({', '.join(sorted(cols - basic))})")

if not issues_found:
    print("Todas las casas tienen mercados básicos completos (H/D/A)")
    if extra_variants:
        print(f"\nVariantes adicionales detectadas:")
        for variant in extra_variants:
            print(f"  • {variant}")

Todas las casas tienen mercados básicos completos (H/D/A)

Variantes adicionales detectadas:
  • PS (PSCA, PSCD, PSCH)


### 5.3 Detección de odds inválidas

Identificación de valores fuera de rango esperado (< 1.0 o > 100).

In [28]:
MIN_VALID_ODD = 1.0   # Cuotas < 1.0 son matemáticamente inválidas
MAX_VALID_ODD = 100.0 # Cuotas > 100 son extremadamente raras en ligas principales

issues_found = False

for s in seasons_sorted:
    df = dfs_all[s]
    for col in odds_columns:
        invalid = ((df[col] < MIN_VALID_ODD) | (df[col] > MAX_VALID_ODD)).sum()
        if invalid > 0:
            print(f"[!] {s} - {col}: {invalid} cuotas fuera de rango [{MIN_VALID_ODD}, {MAX_VALID_ODD}]")
            issues_found = True

if not issues_found:
    print(f"Todas las cuotas están en el rango válido [{MIN_VALID_ODD}, {MAX_VALID_ODD}]")

Todas las cuotas están en el rango válido [1.0, 100.0]


### 5.4 Cobertura de odds por partido

Detección de partidos sin ninguna cuota disponible en el dataset.

In [30]:
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s]
    rows_missing_all_odds = df[odds_columns].isnull().all(axis=1).sum()
    if rows_missing_all_odds > 0:
        print(f"[!] {s}: {rows_missing_all_odds} partidos sin cuotas")
        issues_found = True

if not issues_found:
    print("Todos los partidos tienen al menos una cuota disponible")

Todos los partidos tienen al menos una cuota disponible


## 6) Conclusiones del análisis exploratorio

El dataset estudiado comprende **10 temporadas** (2014/15–2023/24) con un total de **3,800 partidos** de La Liga. 

Se observó **variabilidad en el número de columnas** entre temporadas, principalmente debido a diferencias en la cobertura de casas de apuestas a lo largo del tiempo. A partir de esta variabilidad se definió un **core dataset de 43 variables comunes**, presente de forma consistente en todas las temporadas, que constituye la base estable para el análisis longitudinal.

El análisis de tipos de datos reveló **ausencia total de drift**: las 43 columnas del core mantienen tipos consistentes entre temporadas, con **37 variables numéricas** verificadas como estables.

### Estructura del core dataset

El core dataset se distribuye en:

| Grupo | Variables |
|-------|-----------|
| Identificación (4) | `Date`, `Div`, `HomeTeam`, `AwayTeam` |
| Resultados (6) | `FTHG`, `FTAG`, `FTR`, `HTHG`, `HTAG`, `HTR` |
| Estadísticas (12) | `HS`, `AS`, `HST`, `AST`, `HF`, `AF`, `HC`, `AC`, `HY`, `AY`, `HR`, `AR` |
| Cuotas (21) | `B365`, `BW`, `IW`, `PS`, `VC`, `WH` (H/D/A + variantes) |

### Calidad y completitud

- Se detectaron **633 valores nulos** en el core dataset (**<0.5%**).
- Los missing values se concentran en **variables de cuotas**, especialmente `IWA`, `IWD` e `IWH` en **2023/24 (50%)**.
- Las variables de **resultados y estadísticas de partido** presentan **completitud total**.
- La estrategia recomendada es **priorizar casas con mayor cobertura** y menor proporción de nulos en fases posteriores.

### Requisitos para la fase de limpieza

- Conversión de `Date` a formato `datetime`.
- Codificación de variables categóricas (`HomeTeam`, `AwayTeam`, `FTR`, `HTR`).
- Tratamiento de la variable categórica`Div`: variable identificadora de liga en la fuente original (valor constante `SP1` para La Liga), no aporta información para modelado.
- Se verificó **consistencia total en nombres de equipos**: no se detectaron variaciones tras normalización (espacios, acentos, mayúsculas).
- Incorporación de validaciones automáticas: categorías válidas (`FTR`/`HTR`), ausencia de duplicados, rangos esperados en cuotas y estadísticas.

---

En conjunto, el core dataset muestra estabilidad estructural, coherencia interna y alta completitud, constituyendo una base adecuada para la fase de limpieza y posterior modelado.

---